# Day 1 Data Validation and Email Injection

This notebook documents the Day 1 data-preparation work. Day 1 validates raw WikiNeural examples, repairs BIO issues, creates clean splits, injects synthetic email entities, and writes the artifacts consumed by Day 2 preprocessing.


## Related Day 1 Files

| File | Role |
|---|---|
| `scripts/01_validate_data.py` | Loads raw JSON, validates examples, repairs BIO violations, creates clean parquet splits, writes reports/checksums/histogram. |
| `scripts/01_inject_emails.py` | Injects synthetic email addresses into clean train/validation/test splits and writes injection metadata. |
| `src/pii_masking/day1_data.py` | Reusable validation, splitting, parquet export, checksum, and span-count logic. |
| `src/pii_masking/day1_injection.py` | Reusable synthetic email generation and BIO-safe insertion logic. |
| `data/injection_config.json` | Email domains, insertion contexts, tokenization ratio, and injection-rate configuration. |
| `tests/test_injection.py` | Tests BIO validity, injection rate, domain isolation, and validation helpers. |


## How To Reproduce Day 1

Run from the project root:

```bash
python scripts/01_validate_data.py
python scripts/01_inject_emails.py
```


## Day 1 Outputs

| Artifact | Path |
|---|---|
| Validation report | `data/processed/data_validation_report.json` |
| Clean train split | `data/processed/train_clean.parquet` |
| Clean validation split | `data/processed/val_clean.parquet` |
| Clean test split | `data/processed/test_clean.parquet` |
| Injection report | `data/processed/injection_report.json` |
| Injected train split | `data/processed/train_with_emails.parquet` |
| Injected validation split | `data/processed/val_with_emails.parquet` |
| Injected test split | `data/processed/test_injected.parquet` |
| Raw checksums | `data/checksums.txt` |
| Token-length histogram | `reports/figures/token_length_distribution.png` |


## Summarize Validation Report

Purpose: inspect Day 1 validation output. Critical thinking: verify data quality, valid/invalid row counts, BIO repairs, entity density, and sequence-length assumptions before any modeling.


In [1]:
import json
from pathlib import Path

report_path = Path('../data/processed/data_validation_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    for split_name, split_report in report.items():
        print(f'[{split_name}]')
        print('  total_examples:', split_report.get('total_examples'))
        print('  valid_examples:', split_report.get('valid_examples'))
        print('  invalid_examples:', split_report.get('invalid_examples'))
        print('  bio_violations_fixed:', split_report.get('bio_violations_fixed'))
        print('  length_stats:', split_report.get('length_stats'))
        print('  per_span_counts:', split_report.get('per_span_counts'))
        print('  email_span_count:', split_report.get('email_span_count'))
else:
    print(f'Validation report not found at {report_path}. Run scripts/01_validate_data.py first.')


[train]
  total_examples: 28516
  valid_examples: 28516
  invalid_examples: 0
  bio_violations_fixed: 6
  length_stats: {'min': 2, 'max': 163, 'mean': 24.86, 'median': 23.0, 'p95': 47.0}
  per_span_counts: {'0_per': 0, '1_per': 21572, '2plus_per': 6944}
  email_span_count: 0
[test]
  total_examples: 3650
  valid_examples: 3650
  invalid_examples: 0
  bio_violations_fixed: 3
  length_stats: {'min': 3, 'max': 97, 'mean': 23.84, 'median': 22.0, 'p95': 46.0}
  per_span_counts: {'0_per': 0, '1_per': 2701, '2plus_per': 949}
  email_span_count: 0


## Summarize Email Injection Report

Purpose: inspect Day 1 synthetic-email injection output. Critical thinking: verify injection rate, context distribution, and single-token/multi-token balance before using the data downstream.


In [2]:
import json
from pathlib import Path

injection_report_path = Path('../data/processed/injection_report.json')
if injection_report_path.exists():
    injection_report = json.loads(injection_report_path.read_text(encoding='utf-8'))
    for split_name, split_report in injection_report.items():
        print(f'[{split_name}]')
        print('  total_examples:', split_report.get('total_examples'))
        print('  candidates:', split_report.get('candidates'))
        print('  injected:', split_report.get('injected'))
        print('  injection_rate_actual:', split_report.get('injection_rate_actual'))
        print('  single_token_count:', split_report.get('single_token_count'))
        print('  multi_token_count:', split_report.get('multi_token_count'))
        print('  context_distribution:', split_report.get('context_distribution'))
else:
    print(f'Injection report not found at {injection_report_path}. Run scripts/01_inject_emails.py first.')


[train_val]
  total_examples: 28516
  candidates: 28516
  injected: 17109
  injection_rate_actual: 0.5999789591808108
  single_token_count: 15418
  multi_token_count: 1691
  context_distribution: {'post_name': 8436, 'parenthetical': 5205, 'contact_tail': 3468}
[test]
  total_examples: 3650
  candidates: 3650
  injected: 2190
  injection_rate_actual: 0.6
  single_token_count: 1967
  multi_token_count: 223
  context_distribution: {'post_name': 1125, 'parenthetical': 667, 'contact_tail': 398}


## Notes For Evaluators

- Day 1 implementation is script-based for reproducibility.
- This notebook explains and inspects the saved Day 1 artifacts.
- Day 2 starts from the injected parquet files created here.
